# Exercise 4

1. **Execute o código abaixo em um arquivo cpp da seguinte maneira**:
    
    * Na pasta ompenmp/notebooks/codigo_externo, crie um aquivo chamado: 
        * "exercise_4.cpp".

    * Clique duas vezes para abrí-lo no editor do VSCode.

    * Copie e cole o código abaixo dentro do arquivo e salve.

    * No terminal Linux, vá até a pasta do aquivo: 
        * cd ompenmp/notebooks/codigo_externo

    * Compile o aquivo:
        * g++ -fopenmp exercise_4.cpp -o exercise_4.exe

    * Execute o programa:
        ./exercise_4.exe

2. **Execute o código no notebook (single thread) e depois fora do notebook (número de threads == número de núcleos).**
    * Insira o código necessário para iniciar a contagem do tempo antes do paralelismo
    * Insira o código necessário para calcular o tempo corrigo após o término do paralelismo
    * Faça 3 execuções do código fora do notebook e, para cada uma, faça:
        * **Varie o valor de "n" e o número de threads**    
        * Imprima os resultados no console
        * Copie o tempo para a célula do tipo markdown que se encontra abaixo da célula de código
    * Compare e comente a variação nos tempos.

3. **Volte a este notebook**:
    * Na célula do tipo markdown abaixo da célula que contém o código a ser executado.

**Obs: dentro do notebook é executada apenas uma thread. Por isso, execute fora do notebook para ver o paralelismo.** 


O código abaixo realiza uma multiplicação de matrizes usando OpenMP.

In [ ]:
#include "notebooks_reserved_code/openmp_config.h"

#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

int main() 
{
	const int NRA=62;                 /* number of rows in matrix A */
	const int NCA=15;                 /* number of columns in matrix A */
	const int NCB=7;                  /* number of columns in matrix B */
	
	int	tid, nthreads, i, j, k, chunk;
	double	a[NRA][NCA],           		/* matrix A to be multiplied */
			b[NCA][NCB],           		/* matrix B to be multiplied */
			c[NRA][NCB];           		/* result matrix C */

	chunk = 10;                    

	/*1 - O QUE ESTÁ ACONCENDO AQUI?*/
	#pragma omp parallel shared(a,b,c,nthreads,chunk) private(tid,i,j,k)
	{
		
		tid = omp_get_thread_num();
		if (tid == 0)
		{
			nthreads = omp_get_num_threads();
			printf("Starting matrix multiple example with %d threads\n",nthreads);
			printf("Initializing matrices...\n");
		}
		
		/*2 - O QUE ESTÁ ACONCENDO AQUI?*/
		#pragma omp for schedule (static, chunk) 
		for(i=0; i<NRA; i++){
			for(j=0; j<NCA; j++){
				a[i][j]= i+j;
			}
		}
		/*3 - O QUE ESTÁ ACONCENDO AQUI?*/	
		#pragma omp for schedule (static, chunk)
		for(i=0; i<NCA; i++){
			for(j=0; j<NCB; j++){
			  b[i][j]= i*j;
			}
		}
		/*4 - O QUE ESTÁ ACONCENDO AQUI?*/
		#pragma omp for schedule (static, chunk)
		for(i=0; i<NRA; i++){
			for(j=0; j<NCB; j++){
				c[i][j]= 0;
			}
		}

		/*5 - O QUE ESTÁ ACONCENDO AQUI?*/		
		printf("Thread %d starting matrix multiply...\n",tid);
		#pragma omp for schedule (static, chunk)
		for (i=0; i<NRA; i++){
			printf("Thread=%d did row=%d\n",tid,i);
			for(j=0; j<NCB; j++){
			  for (k=0; k<NCA; k++){
				c[i][j] += a[i][k] * b[k][j];
			  }
			}
		}
	}   /*** End of parallel region ***/

	/*** Print results ***/
	printf("******************************************************\n");
	printf("Result Matrix:\n");
	for(i=0; i<NRA; i++){
		for (j=0; j<NCB; j++){ 
			printf("%6.2f   ", c[i][j]);
		}
	  printf("\n"); 
	}
	printf("******************************************************\n");
	printf ("Done.\n");

}

main()

input_line_8:1:10: fatal error: 'openmp_config.h' file not found
#include "openmp_config.h"
         ^~~~~~~~~~~~~~~~~


Interpreter Error: 

## Resultados:
**Obs: após editar uma célula do tipo markdown, é preciso pressionar as tecla CTRL+ENTER para renderizar o texto**


1. Coloque aqui as variações nos tempos de execução para cada variação de "n" e comente o que você obervou com base na matéria estudada e nos materiais de referência.


    | Threads | NRA | NCA | NCB | Tempo (segundos) |
    |---------|-----|-----|-----|------------------|
    | 4       | 62  | 15  | 7   | 0.002794         |
    | 8       | 124 | 30  | 14  | 0.006698         |
    | 2       | 31  | 7   | 3   | 0.001122         |

    O tempo de execução aumenta conforme o tamanho das matrizes cresce, devido à maior complexidade do cálculo da multiplicação. O uso de mais threads traz ganhos em problemas maiores, mas para cargas pequenas o overhead de paralelização pode reduzir a eficiência. Portanto, é importante ajustar o número de threads de acordo com o tamanho do problema para otimizar o desempenho e evitar desperdício de recursos.



2. Observe que, no código, há comentários numerados, como este: 1 - O QUE ESTÁ ACONCENDO AQUI?. Para cada comentário, explique abaixo o que está acontecendo no respectivo código ao qual o comentário se refere.
 
    a. Comentário 1:  
    Está criando uma região paralela com a diretiva `#pragma omp parallel`, onde as variáveis `a, b, c, nthreads, chunk` são compartilhadas entre as threads, e as variáveis `tid, i, j, k` são privadas para cada thread.

    b. Comentário 2:  
    A cláusula `schedule` define como as iterações do loop são distribuídas entre as threads na paralelização, controlando quando e quantas iterações cada thread vai executar e a estratégia de divisão do trabalho. Sendo assim, no trecho indicado, as iterações são divididas em blocos fixos de tamanho definido pelo chunk, e distribuídas entre as threads de forma estática. Cada thread inicializa os elementos de suas linhas atribuídos, preenchendo a matriz com o valor da soma dos índices das linhas e colunas. 

    b. Comentário 3:  
    Está acontecendo a mesma coisa que a anterior, mas com outra matriz (`b`).

    b. Comentário 4:  
    Nesse trecho, cada thread recebe blocos de linhas para inicializar e zera os elementos correspondentes da matriz `c`. Assim, a matriz resultado é preparada para receber os valores da multiplicação.

    b. Comentário 5:  
    Neste trecho, cada thread imprime uma mensagem indicando que está iniciando a multiplicação. Em seguida, o loop externo é paralelizado com divisão estática em blocos, onde cada thread calcula as linhas da matriz resultado `c` que lhe foram atribuídas. Para cada elemento `c[i][j]`, a thread realiza a soma dos produtos correspondentes das linhas de `a` e colunas de `b`, completando assim a multiplicação.

## Entrega:

**Salve este notebook com as suas respostas e poste como entrega da atividade no Canvas.**